<a href="https://colab.research.google.com/github/joshuajhchoi/ai2learn/blob/master/Multimodal_Age_and_Gender_Classification_Using_Ear_and_Profile_Face_Images.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torchvision import transforms
from PIL import Image

class EarProfileDataset(Dataset):
    def __init__(self, root_dir, csv_file, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.data = pd.read_csv(csv_file)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img_path = os.path.join(self.root_dir, self.data.loc[idx, 'filename'])
        img = Image.open(img_path).convert('RGB')
        label_age = self.data.loc[idx, 'age']
        label_gender = self.data.loc[idx, 'gender']
        sample = {'image': img, 'age': label_age, 'gender': label_gender}
        if self.transform:
            sample['image'] = self.transform(sample['image'])
        return sample

class AgeGenderNet(nn.Module):
    def __init__(self):
        super(AgeGenderNet, self).__init__()
        self.ear_conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.ear_bn1 = nn.BatchNorm2d(32)
        self.ear_conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.ear_bn2 = nn.BatchNorm2d(64)
        self.ear_conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.ear_bn3 = nn.BatchNorm2d(128)
        self.ear_fc1 = nn.Linear(128 * 16 * 16, 256)
        self.ear_bn4 = nn.BatchNorm1d(256)
        self.profile_conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.profile_bn1 = nn.BatchNorm2d(32)
        self.profile_conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.profile_bn2 = nn.BatchNorm2d(64)
        self.profile_conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.profile_bn3 = nn.BatchNorm2d(128)
        self.profile_fc1 = nn.Linear(128 * 16 * 16, 256)
        self.profile_bn4 = nn.BatchNorm1d(256)
        self.fc1 = nn.Linear(512, 256)
        self.bn1 = nn.BatchNorm1d(256)
        self.fc2_age = nn.Linear(256, 1)
        self.fc2_gender = nn.Linear(256, 1)

    def forward(self, x_ear, x_profile):
        x_ear = self.ear_conv1(x_ear)
        x_ear = nn.functional.relu(self.ear_bn1(x_ear))
        x_ear = nn.functional.max_pool2d(x_ear, 2)
        x_ear = self.ear_conv2(x_ear)
        x_ear = nn.functional.relu(self.ear_bn2(x_ear))
        x_ear = nn.functional.max_pool2d(x_ear, 2)
        x_ear = self.ear_conv3(x_ear)
        x_ear = nn.functional.relu(self.ear_bn3(x_ear))
        x_ear = x_ear.view(-1, 128 * 16 * 16)
        x_ear = self.ear_fc1(x_ear)
        x_ear = nn.functional.relu(self.ear_bn4(x_ear))




# Define data loading and preprocessing steps
data_transforms = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])
train_dataset = EarProfileDataset(root_dir='path/to/train/images',
                                  csv_file='path/to/train/csv',
                                  transform=data_transforms)
train_dataloader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)

# Initialize the model, optimizer, and loss function
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = AgeGenderNet().to(device)
optimizer = optim.Adam(model.parameters(), lr=0.001)
criterion_age = nn.MSELoss()
criterion_gender = nn.BCEWithLogitsLoss()

# Training loop
num_epochs = 10
for epoch in range(num_epochs):
    running_loss_age = 0.0
    running_loss_gender = 0.0
    for i, batch in enumerate(train_dataloader):
        # Move data to device
        images = batch['image'].to(device)
        labels_age = batch['age'].float().to(device)
        labels_gender = batch['gender'].float().to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward pass
        outputs_age, outputs_gender = model(images[:, :3, :, :], images[:, 3:, :, :])
        loss_age = criterion_age(outputs_age.squeeze(), labels_age)
        loss_gender = criterion_gender(outputs_gender.squeeze(), labels_gender)
        loss = loss_age + loss_gender

        # Backward pass and optimization
        loss.backward()
        optimizer.step()

        # Update running losses
        running_loss_age += loss_age.item()
        running_loss_gender += loss_gender.item()

    # Print epoch statistics
    epoch_loss_age = running_loss_age / len(train_dataloader)
    epoch_loss_gender = running_loss_gender / len(train_dataloader)
    print(f'Epoch {epoch+1}/{num_epochs}: '
          f'Age Loss: {epoch_loss_age:.4f}, '
          f'Gender Loss: {epoch_loss_gender:.4f}')

# Define the validation dataset and dataloader
val_dataset = EarProfileDataset(root_dir='path/to/val/images',
                                csv_file='path/to/val/csv',
                                transform=data_transforms)
val_dataloader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)

# Evaluate the model on the validation set
model.eval()
with torch.no_grad():
    running_loss_age = 0.0
    running_loss_gender = 0.0
    for i, batch in enumerate(val_dataloader):
        # Move data to device
        images = batch['image'].to(device)
        labels_age = batch['age'].float().to(device)
        labels_gender = batch['gender'].float().to(device)

        # Forward pass
        outputs_age, outputs_gender = model(images[:, :3, :, :], images[:, 3:, :, :])
        loss_age = criterion_age(outputs_age.squeeze(), labels_age)
        loss_gender = criterion_gender(outputs_gender.squeeze(), labels_gender)

        # Update running losses
        running_loss_age += loss_age.item()
        running_loss_gender += loss_gender.item()

    # Print validation statistics
    val_loss_age = running_loss_age / len(val_dataloader)
    val_loss_gender = running_loss_gender / len(val_dataloader)
    print(f'Validation Loss: '
          f'Age Loss: {val_loss_age:.4f}, '
          f'Gender Loss: {val_loss_gender:.4f}')

# Define the test dataset and dataloader
test_dataset = EarProfileDataset(root_dir='path/to/test/images',
                                 csv_file='path/to/test/csv',
                                 transform=data_transforms)
test_dataloader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=4)

# Make predictions on test images
model.eval()
with torch.no_grad():
    for i, batch in enumerate(test_dataloader):
        # Move data to device
        images = batch['image'].to(device)

        # Forward pass
        outputs_age, outputs_gender = model(images[:, :3, :, :], images[:, 3:, :, :])

        # Extract predicted age and gender
        predicted_age = int(round(outputs_age.item()))
        predicted_gender = 'Male' if outputs_gender.item() > 0 else 'Female'

        # Print results
        print(f'Test Image {i+1}: Predicted Age: {predicted_age}, Predicted Gender: {predicted_gender}')


FileNotFoundError: ignored